In [1]:
model_id = 'fine-tuning-sherlock'

In [ ]:
from ast import literal_eval
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, classification_report

from sherlock.deploy.model import SherlockModel

## Load dataset for training, validation, testing


### Train

In [3]:
start = datetime.now()
print(f'Started at {start}')

X_train = pd.read_parquet('../custom_data/processed/train.parquet')
y_train = pd.read_parquet('../custom_data/raw/train_labels.parquet').values.flatten()

y_train = np.array([x.lower() for x in y_train])

print(f'Load data (train) process took {datetime.now() - start} seconds.')

Started at 2025-05-12 13:43:36.989171
Load data (train) process took 0:00:00.210713 seconds.


In [4]:
print('Distinct types for columns in the Dataframe (should be all float32):')
print(set(X_train.dtypes))

Distinct types for columns in the Dataframe (should be all float32):
{dtype('float32')}


### Validation

In [5]:
start = datetime.now()
print(f'Started at {start}')

X_validation = pd.read_parquet('../custom_data/processed/validation.parquet')
y_validation = pd.read_parquet('../custom_data/raw/validation_labels.parquet').values.flatten()

y_validation = np.array([x.lower() for x in y_validation])

print(f'Load data (validation) process took {datetime.now() - start} seconds.')

Started at 2025-05-12 13:43:38.579242
Load data (validation) process took 0:00:00.079802 seconds.


### Test

In [6]:
start = datetime.now()
print(f'Started at {start}')

X_test = pd.read_parquet('../custom_data/processed/test.parquet')
y_test = pd.read_parquet('../custom_data/raw/test_labels.parquet').values.flatten()

y_test = np.array([x.lower() for x in y_test])

print(f'Finished at {datetime.now()}, took {datetime.now() - start} seconds')

Started at 2025-05-12 13:43:39.828493
Finished at 2025-05-12 13:43:39.902127, took 0:00:00.073643 seconds


# Initialize the model

### Load Sherlock weights

In [7]:
model = SherlockModel();
model.initialize_model_from_json(with_weights=True, model_id=model_id);

NameError: name 'SherlockModel' is not defined

In [ ]:
from sherlock.model import ColumnNet          # or your wrapper class
import torch, torch.nn as nn

# 1. Load the pretrained model
ckpt = torch.load("checkpoints/colnet.pt", map_location="cpu")
model = ColumnNet(n_classes=78)               # same size as checkpoint
model.load_state_dict(ckpt["state_dict"], strict=False)

# 2. Replace the final layer
in_dim = model.fc.in_features                 # 1024
model.fc = nn.Linear(in_dim, num_labels)      # your label count

# 3. Freeze everything except the new layer
for p in model.parameters():
    p.requires_grad_(False)
for p in model.fc.parameters():
    p.requires_grad_(True)

# 4. Optimiser & loss
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=3e-4, weight_decay=1e-4)
criterion  = FocalLoss(gamma=2.0)             # or weighted CE